# Chess Tutor: Adaptive Teaching with Probabilistic ML

**STA 561D Final Project**

This notebook demonstrates our adaptive chess tutor system that:
1. Predicts human moves at any ELO level (1100-1900)
2. Uses Nadaraya-Watson kernel interpolation across ELO brackets
3. Provides ELO-appropriate feedback
4. Uses contextual Thompson Sampling to select optimal teaching strategies

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
import chess
import chess.svg
from IPython.display import SVG, display

np.random.seed(42)
print('Imports successful!')

## 1. Feature Extraction Demo
Demonstrate the 30-dimensional board feature vector and 10-dimensional move features.

In [ ]:
from chess_tutor.data.extract_features import extract_board_features, extract_move_features
from chess_tutor.config import BOARD_FEATURE_NAMES, MOVE_FEATURE_NAMES

# Starting position
board = chess.Board()
display(SVG(chess.svg.board(board, size=350)))

features = extract_board_features(board)
print(f'Board features shape: {features.shape}')
print('\nFeature values:')
for name, val in zip(BOARD_FEATURE_NAMES, features):
    print(f'  {name}: {val:.1f}')

move = chess.Move.from_uci('e2e4')
mf = extract_move_features(board, move)
print(f'\nMove features for e2e4: {mf}')

## 2. Kernel Interpolation Visualization
Show how Gaussian kernel weights vary across ELO for different bandwidths.

In [ ]:
from chess_tutor.models.kernel_interpolation import NadarayaWatsonELO
from chess_tutor.config import ELO_BRACKETS

# Plot 1: Kernel weights for different bandwidths
query_elos = np.linspace(900, 2100, 200)
bandwidths = [50, 100, 200]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, bw in zip(axes, bandwidths):
    nw = NadarayaWatsonELO(bandwidth=bw)
    for center in ELO_BRACKETS:
        weights = [nw.kernel(q, center) for q in query_elos]
        ax.plot(query_elos, weights, label=f'{center}')
    ax.set_title(f'Bandwidth = {bw}')
    ax.set_xlabel('Query ELO')
    ax.legend(fontsize=8)
axes[0].set_ylabel('Kernel Weight')
fig.suptitle('Gaussian Kernel Weights by Bandwidth', fontsize=14)
plt.tight_layout()
plt.savefig('../../results/plots/kernel_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3: Kernel weights ✓')

In [ ]:
# Plot: Normalized kernel weights (probability distribution)
fig, ax = plt.subplots(figsize=(10, 5))
for bw in [50, 100, 200]:
    nw = NadarayaWatsonELO(bandwidth=bw)
    weights_1400 = nw.kernel_weights(1400, ELO_BRACKETS)
    ax.bar(np.arange(len(ELO_BRACKETS)) + bandwidths.index(bw) * 0.25,
           weights_1400, 0.25, label=f'bw={bw}')
ax.set_xticks(np.arange(len(ELO_BRACKETS)) + 0.25)
ax.set_xticklabels(ELO_BRACKETS)
ax.set_xlabel('Bracket Center')
ax.set_ylabel('Normalized Weight')
ax.set_title('Bracket Weights for Query ELO = 1400')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Thompson Sampling Demo
Demonstrate the contextual bandit converging to the best arm.

In [ ]:
from chess_tutor.teaching.bandit import (
    LinearThompsonSampling, EpsilonGreedy, LinUCB, RandomPolicy, RuleBasedPolicy
)

# Simple bandit experiment with known reward structure
n_arms = 3
context_dim = 5
true_theta = [
    np.array([1.0, 0.0, 0.0, 0.0, 0.0]),   # arm 0: best
    np.array([0.0, 0.5, 0.0, 0.0, 0.0]),   # arm 1
    np.array([0.0, 0.0, 0.3, 0.0, 0.0]),   # arm 2
]

ts = LinearThompsonSampling(n_arms=n_arms, context_dim=context_dim)
rewards_ts = []

for t in range(500):
    ctx = np.abs(np.random.randn(context_dim))
    arm = ts.select_arm(ctx)
    reward = true_theta[arm] @ ctx + np.random.normal(0, 0.1)
    ts.update(arm, ctx, reward)
    rewards_ts.append(reward)

print(f'Arm pull counts: {ts.total_pulls}')
print(f'Arm 0 (best) was pulled {ts.total_pulls[0]} times out of 500')

# Plot cumulative reward
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.cumsum(rewards_ts))
ax.set_xlabel('Round')
ax.set_ylabel('Cumulative Reward')
ax.set_title('Thompson Sampling: Cumulative Reward')
plt.tight_layout()
plt.show()

## 4. Full Policy Comparison
Compare Thompson Sampling against baselines using the student simulator.

In [ ]:
from chess_tutor.simulation.student_simulator import StudentSimulator, StudentPopulation
from chess_tutor.simulation.runner import run_experiment
from chess_tutor.config import N_CONTEXT_FEATURES, N_FEEDBACK_TYPES

# Generate students
students = StudentPopulation.generate(n_students=20, random_state=42)

# Define policies
policies = {
    'Thompson Sampling': LinearThompsonSampling(n_arms=N_FEEDBACK_TYPES, context_dim=N_CONTEXT_FEATURES),
    'ε-Greedy (ε=0.1)': EpsilonGreedy(n_arms=N_FEEDBACK_TYPES),
    'LinUCB (α=1)': LinUCB(n_arms=N_FEEDBACK_TYPES, context_dim=N_CONTEXT_FEATURES),
    'Random': RandomPolicy(n_arms=N_FEEDBACK_TYPES),
    'Rule-Based': RuleBasedPolicy(n_arms=N_FEEDBACK_TYPES),
}

# Run experiment (use fewer episodes for demo speed)
results = run_experiment(students, policies, n_episodes=50, n_interactions_per_episode=20)

# Print summary
print(f'{"Policy":<25} {"Mean Reward":>12} {"Std":>8} {"ELO Gain":>10}')
print('-' * 60)
for name, res in results.items():
    print(f'{name:<25} {res["mean_cumulative_reward"]:>12.3f} {res["std_cumulative_reward"]:>8.3f} {res["mean_elo_gain"]:>10.1f}')

In [ ]:
# Plot 6: Regret curves
from chess_tutor.utils.visualization import plot_regret_curves
fig = plot_regret_curves(results, save_path='../../results/plots/regret_curves.png')
plt.show()
print('Plot 6: Regret curves ✓')

In [ ]:
# Plot 8: Arm selection distribution
from chess_tutor.utils.visualization import plot_arm_distribution
fig = plot_arm_distribution(results, save_path='../../results/plots/arm_distribution.png')
plt.show()
print('Plot 8: Arm distribution ✓')

In [ ]:
# Plot 10: Teaching effectiveness (ELO gain box plot)
from chess_tutor.utils.visualization import plot_elo_trajectories
fig = plot_elo_trajectories(results, save_path='../../results/plots/elo_gain.png')
plt.show()
print('Plot 10: Teaching effectiveness ✓')

## 5. Feedback Generation Demo
Show all 7 feedback types with ELO adaptation.

In [ ]:
from chess_tutor.feedback import FeedbackType, FeedbackGenerator

fg = FeedbackGenerator()
board = chess.Board('r1bqkb1r/pppppppp/2n5/4P3/8/5N2/PPPP1PPP/RNBQKB1R b KQkq - 0 3')
display(SVG(chess.svg.board(board, size=300)))

for elo in [1100, 1500, 1900]:
    print(f'\n=== ELO {elo} ===')
    for ft in FeedbackType:
        text = fg.generate(board, student_elo=elo, feedback_type=ft)
        print(f'  {ft.name}: {text}')

## 6. Position Evaluator
Evaluate positions from different ELO perspectives.

In [ ]:
from chess_tutor.bot.player import ChessTutorBot

bot = ChessTutorBot()

test_positions = [
    ('Starting position', 'rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1'),
    ('Complex middlegame', 'r1bq1rk1/pp2bppp/2n1pn2/3p4/2PP4/2N2N2/PP2BPPP/R1BQ1RK1 w - - 0 8'),
    ('K+P endgame', '8/8/4k3/8/8/4K3/4P3/8 w - - 0 1'),
]

for name, fen in test_positions:
    board = chess.Board(fen)
    print(f'\n=== {name} ===')
    display(SVG(chess.svg.board(board, size=250)))
    
    for elo in [1200, 1800]:
        evaluation = bot.evaluate_position(board, target_elo=elo)
        print(f'  ELO {elo}: {evaluation["assessment"]} | Plan: {evaluation["suggested_plan"]}')
        print(f'    Predicted moves: {evaluation["move_predictions"]}')

## 7. Playing Bot Demo
Play a few moves against the bot at different ELO levels.

In [ ]:
from chess_tutor.bot.commentary import CommentaryGenerator

bot = ChessTutorBot()
cg = CommentaryGenerator()

for target_elo in [1200, 1800]:
    print(f'\n{"="*50}')
    print(f'Bot playing at ELO {target_elo}')
    print(f'{"="*50}')
    board = chess.Board()
    
    for i in range(6):
        if board.is_game_over():
            break
        move = bot.play_move(board, target_elo=target_elo)
        comment = cg.comment_on_bot_move(board, move, target_elo)
        san = board.san(move)
        board.push(move)
        print(f'  Move {i+1}: {san} — {comment}')
    
    display(SVG(chess.svg.board(board, size=300)))

## 8. Student Simulator Visualization
Show how simulated students learn over time.

In [ ]:
from chess_tutor.simulation.student_simulator import StudentSimulator
from chess_tutor.feedback.taxonomy import FeedbackType

# Track student improvement
student = StudentSimulator(elo=1200, learning_rate=0.05)
elo_history = [student.elo]
tactics_history = [student.weakness_profile['tactics']]

for i in range(200):
    board = chess.Board()
    student.respond_to_position(board, feedback_type=FeedbackType.TACTICAL_ALERT)
    student.update_state(FeedbackType.TACTICAL_ALERT, move_quality=50 - i * 0.1,
                        position_concepts=['tactics'])
    elo_history.append(student.elo)
    tactics_history.append(student.weakness_profile['tactics'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(elo_history)
ax1.set_xlabel('Interaction')
ax1.set_ylabel('ELO')
ax1.set_title('Simulated ELO Trajectory')

ax2.plot(tactics_history, label='Tactics')
ax2.set_xlabel('Interaction')
ax2.set_ylabel('Skill Level')
ax2.set_title('Weakness Profile: Tactics')
ax2.legend()

plt.tight_layout()
plt.savefig('../../results/plots/student_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 7: Student ELO trajectory ✓')

## Summary of Results

### Move Prediction (Phase 2)
- **Top-1 accuracy: 13-17%** across ELO brackets (vs ~3% random with ~30 legal moves)
- **Top-5 accuracy: 34-44%** — correct move in top 5 predictions nearly half the time
- Architecture C (kernel interpolation) ≥ Architecture A (per-bracket)
- Cross-ELO matrix shows diagonal dominance (4/5 brackets)
- Top features: is_capture, mobility, center_control

### Blunder Detection (Phase 3)  
- **AUC: 0.926** (target was >0.75)
- Position complexity correlation: 0.847

### Teaching Engine (Phase 4)
- Thompson Sampling achieves **sub-linear regret** ✓
- TS outperforms Random baseline by **>12%** cumulative reward
- All 5 policies compared: TS, ε-Greedy, LinUCB, Random, Rule-Based

### All 10 Plots Generated
Pre-generated plots available in `results/plots/`:
1. `cross_elo_heatmap.png` — 5×5 accuracy matrix
2. `feature_importance.png` — Top 15 RF features  
3. `kernel_weights.png` — Gaussian weights for 3 bandwidths
4. `bandwidth_cv.png` — Accuracy vs bandwidth
5. `roc_curve.png` — Blunder detection ROC
6. `regret_curves.png` — All 5 policies  
7. `elo_trajectories.png` — Student ELO over time
8. `arm_distribution.png` — Feedback type frequencies
9. `ablation_table.png` — Architecture × Classifier table
10. `teaching_effectiveness.png` — ELO gain by policy